In [ ]:
# Day 2 · vLLM on T4.  The old `pip install vllm==0.7.3` failed because it fought Kaggle's
# torch 2.10 / transformers 5.0. vLLM now lives in its own venv (scripts/vllm_setup.sh).
!git clone https://github.com/Maheendra-mj/3-day-inference-challenge 2>/dev/null || git -C 3-day-inference-challenge pull
%cd /kaggle/working/3-day-inference-challenge

In [ ]:
# 1) Install vLLM + infer-lab into /tmp/vllm-env (~3-5 min, once per session)
!bash scripts/vllm_setup.sh

In [ ]:
# 2) Smoke test: does vLLM run on sm_75? Which attention backend? How big is the KV cache?
!/tmp/vllm-env/bin/python scripts/vllm_smoke.py > /tmp/vllm_smoke.log 2>&1; echo "exit code: $?"
!grep -i -E "backend|kv cache|cuda graph|concurrency" /tmp/vllm_smoke.log | cut -c1-220
!tail -n 25 /tmp/vllm_smoke.log

In [ ]:
# 3) Only after the smoke test passes: vLLM offline, same sweep as the Day-1 baseline (CUDA graphs ON)
!/tmp/vllm-env/bin/infer-lab bench --backend vllm_offline --config configs/vllm_offline.yaml --tag vllm 2>&1 | grep -E "infer_lab|Error|Traceback"

In [ ]:
# 4) vLLM with CUDA graphs OFF: separates "launch floor removed" from "better attention kernels"
!/tmp/vllm-env/bin/infer-lab bench --backend vllm_offline --config configs/vllm_offline_eager.yaml --tag vllm_eager 2>&1 | grep -E "infer_lab|Error|Traceback"

In [ ]:
# 5) HF eager baseline on THIS VM (system python, Kaggle's torch), so every row in the
#    comparison comes from the same machine: launch-bound numbers vary across VMs (DAY1 §8.6)
!pip install -q -e . && infer-lab bench --config configs/bench.yaml --tag hf_baseline 2>&1 | grep -E "infer_lab|Error|Traceback"

In [ ]:
# 6) Side-by-side: hf_baseline vs vllm_eager vs vllm
!infer-lab report